In [1]:
import os
import pickle
import pycisTopic
import scanpy as sc
import pandas as pd
from pycisTopic.cistopic_class import *


workdir = "./ComicGTN/data/ARPKD_kidney"
out_dir = "./ComicGTN/data/ARPKD_kidney/SCENICplus_outs"
os.makedirs(out_dir, exist_ok = True)
fragments_dict = {"Epi-17": "./ComicGTN/data/ARPKD_kidney/Epi-17/fragments.tsv.gz",
                              "Cor-15": "./ComicGTN/data/ARPKD_kidney/Cor-15/fragments.tsv.gz",
                              "Epi-17-KO": "./ComicGTN/data/ARPKD_kidney/Epi-17-KO/fragments.tsv.gz"}


Cell_names = pd.read_csv(os.path.join(workdir, "Cell_names.tsv"), sep = "\t", header = None)
Leiden_labels = pd.read_csv(os.path.join(workdir, "ComicGTN_pred.tsv"), sep = "\t", header = None)
Cell_types = pd.read_csv(os.path.join(workdir, "Cell_types.tsv"), sep = "\t", header = None)
Sample_ids = pd.read_csv(os.path.join(workdir, "Sample_names.tsv"), sep = "\t", header = None)
Peak_names = pd.read_csv(os.path.join(workdir, "Peak_names.tsv"), sep = "\t", header = None)


cell_data = pd.concat([Cell_types.set_axis(Cell_names[0], axis = 0)[0].rename("Cell_types"),
                                     Leiden_labels.set_axis(Cell_names[0], axis = 0)[0].rename("Leiden_labels"),
                                     Sample_ids.set_axis(Cell_names[0], axis = 0)[0].rename("sample_id")], axis = 1)
cell_data.index.name = None
cell_data["sample_id"] = cell_data["sample_id"].replace({"E17": "Epi-17", "C15": "Cor-15", "E173A": "Epi-17-KO"})
cell_data["Cell_types"] = cell_data["Cell_types"].replace({"STRO 1": "STRO1", "STRO 2": "STRO2", "STRO 3": "STRO3", "STRO 4": "STRO4",
                                                                                          "STRO 5": "STRO5", "STRO 6": "STRO6", "STRO 7": "STRO7", "STRO 8": "STRO8",
                                                                                          "STRO 9": "STRO9", "GLOM 1": "GLOM1", "GLOM 2": "GLOM2", "GLOM 3": "GLOM3",
                                                                                          "GLOM 4": "GLOM4", "PT 1": "PT1", "PT 2": "PT2", "PT 3": "PT3", "DE 1": "DE1",
                                                                                          "DE 2": "DE2", "NEURAL 1": "NEURAL1", "NEURAL 2": "NEURAL2", "EC": "EC"})
cell_data.to_csv(os.path.join(workdir, "cell_data.csv"), index = True, header = True, sep = ",")


ATAC_seq = sc.read(os.path.join(workdir, "Peak_Cell.mtx"))
ATAC_matrix = ATAC_seq.X
cell_data =  pd.read_csv(os.path.join(workdir, "cell_data.csv"), header = 0, index_col = 0)
cistopic_obj = create_cistopic_object(fragment_matrix = ATAC_matrix, path_to_fragments = fragments_dict,
                                                            cell_names = cell_data.index.tolist(), region_names = Peak_names[0].tolist())
cistopic_obj.add_cell_data(cell_data)
new_index = cistopic_obj.cell_data.index.str[:18] + "-" + cistopic_obj.cell_data["sample_id"]
cistopic_obj.cell_data.index = new_index
cistopic_obj.cell_names = cistopic_obj.cell_data.index
pickle.dump(cistopic_obj, open(os.path.join(out_dir, "cistopic_obj.pkl"), "wb"))

In [2]:
from pycisTopic.lda_models import run_cgs_models_mallet
from pycisTopic.lda_models import evaluate_models
from pycisTopic.topic_binarization import binarize_topics
from pycisTopic.topic_qc import compute_topic_metrics, plot_topic_qc, topic_annotation
import matplotlib.pyplot as plt
from pycisTopic.utils import fig2img


os.environ["MALLET_MEMORY"] = "100G"
mallet_path = "/home/jsl/YBR/Mallet-202108/bin/mallet"
models = run_cgs_models_mallet(cistopic_obj, n_topics = [2, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50], n_cpu = 24, n_iter = 500, 
                                                       random_state = 555, alpha = 50, alpha_by_topic = True, eta = 0.1, eta_by_topic = False,
                                                       tmp_path = "./ComicGTN/data/ARPKD_kidney/mallet_tutorial", save_path = "./ComicGTN/data/ARPKD_kidney/mallet_tutorial",
                                                       mallet_path = mallet_path)
model = evaluate_models(models, select_model = 50, return_model = True)
pickle.dump(models, open(os.path.join(out_dir, "models.pkl"), "wb"))
cistopic_obj.add_LDA_model(model)
pickle.dump(cistopic_obj, open(os.path.join(out_dir, "cistopic_obj.pkl"), "wb"))


with open("./ComicGTN/data/ARPKD_kidney/SCENICplus_outs/data/cistopic_obj.pkl", "rb") as f:
    cistopic_obj = pickle.load(f)
region_bin_topics_top_3k = binarize_topics(cistopic_obj, method = "ntop", ntop = 3_000, plot = True, num_columns = 5)
region_bin_topics_otsu = binarize_topics(cistopic_obj, method = "otsu", plot = True, num_columns = 5)
binarized_cell_topic = binarize_topics(cistopic_obj, target = "cell", method = "li", plot = True, num_columns = 5, nbins = 100)


topic_qc_metrics = compute_topic_metrics(cistopic_obj)
fig_dict = {}
fig_dict["CoherenceVSAssignments"] = plot_topic_qc(topic_qc_metrics, var_x = "Coherence", var_y = "Log10_Assignments", 
                                                                                      var_color = "Gini_index", plot = False, return_fig = True)
fig_dict["AssignmentsVSCells_in_bin"] = plot_topic_qc(topic_qc_metrics, var_x = "Log10_Assignments", var_y = "Cells_in_binarized_topic", 
                                                                                       var_color = "Gini_index", plot = False, return_fig = True)
fig_dict["CoherenceVSCells_in_bin"] = plot_topic_qc(topic_qc_metrics, var_x = "Coherence", var_y = "Cells_in_binarized_topic", 
                                                                                    var_color = "Gini_index", plot = False, return_fig = True)
fig_dict["CoherenceVSRegions_in_bin"] = plot_topic_qc(topic_qc_metrics, var_x = "Coherence", var_y = "Regions_in_binarized_topic", 
                                                                                         var_color = "Gini_index", plot = False, return_fig = True)
fig_dict["CoherenceVSMarginal_dist"] = plot_topic_qc(topic_qc_metrics, var_x = "Coherence", var_y = "Marginal_topic_dist", 
                                                                                       var_color = "Gini_index", plot = False, return_fig = True)
fig_dict["CoherenceVSGini_index"] = plot_topic_qc(topic_qc_metrics, var_x = "Coherence", var_y = "Gini_index", 
                                                                                  var_color = "Gini_index", plot = False, return_fig = True)


fig = plt.figure(figsize = (40, 43))
i = 1

for fig_ in fig_dict.keys():
    plt.subplot(2, 3, i)
    img = fig2img(fig_dict[fig_])
    plt.imshow(img)
    plt.axis("off")
    i += 1

plt.subplots_adjust(wspace = 0, hspace = -0.70)
plt.show()


topic_annot = topic_annotation(cistopic_obj, annot_var = "Cell_types", binarized_cell_topic = binarized_cell_topic, general_topic_thr = 0.2)

In [3]:
import numpy as np
from pycisTopic.diff_features import (impute_accessibility, normalize_scores, find_highly_variable_features, find_diff_features)
from pycisTopic.utils import region_names_to_coordinates


imputed_acc_obj = impute_accessibility(cistopic_obj, selected_cells = None, selected_regions = None, scale_factor = 10**6)
normalized_imputed_acc_obj = normalize_scores(imputed_acc_obj, scale_factor = 10**4)
variable_regions = find_highly_variable_features(normalized_imputed_acc_obj, min_disp = 0.05, min_mean = 0.0125, max_mean = 3,
                                                                              max_disp = np.inf, n_bins = 20, n_top_features = None, plot = True)
markers_dict = find_diff_features(cistopic_obj, imputed_acc_obj, variable = "Cell_types", var_features = variable_regions,
                                                      contrasts = None, adjpval_thr = 0.05, log2fc_thr = np.log2(1.5), n_cpu = 5, _temp_dir = "/tmp/ray_spill")


os.makedirs(os.path.join(out_dir, "region_sets"), exist_ok = True)
os.makedirs(os.path.join(out_dir, "region_sets", "Topics_otsu"), exist_ok = True)
os.makedirs(os.path.join(out_dir, "region_sets", "Topics_top_3k"), exist_ok = True)
os.makedirs(os.path.join(out_dir, "region_sets", "DARs_cell_type"), exist_ok = True)

for topic in region_bin_topics_otsu:
    region_names_to_coordinates(region_bin_topics_otsu[topic].index).sort_values(["Chromosome", "Start", "End"]).to_csv(
                                                     os.path.join(out_dir, "region_sets", "Topics_otsu", f"{topic}.bed"), sep = "\t",
                                                     header = False, index = False)
    
for topic in region_bin_topics_top_3k:
    region_names_to_coordinates(region_bin_topics_top_3k[topic].index).sort_values(["Chromosome", "Start", "End"]).to_csv(
                                                     os.path.join(out_dir, "region_sets", "Topics_top_3k", f"{topic}.bed"), sep = "\t",
                                                     header = False, index = False)
    
for cell_type in markers_dict:
    region_names_to_coordinates(markers_dict[cell_type].index).sort_values(["Chromosome", "Start", "End"]).to_csv(
                                                     os.path.join(out_dir, "region_sets", "DARs_cell_type", f"{cell_type}.bed"), sep = "\t",
                                                     header = False, index = False)

In [4]:
import anndata as ad
import pyranges as pr
from pycisTopic.gene_activity import get_gene_activity


chromsizes = pd.read_table(os.path.join(out_dir, "qc", "hg38.chrom_sizes_and_alias.tsv"))
chromsizes.rename({"# ucsc": "Chromosome", "length": "End"}, axis = 1, inplace = True)
chromsizes["Start"] = 0
chromsizes = pr.PyRanges(chromsizes[["Chromosome", "Start", "End"]])
pr_annotation = pd.read_table(os.path.join(out_dir, "qc", "tss.bed")).rename({"Name": "Gene", "# Chromosome": "Chromosome"}, axis = 1)
pr_annotation["Transcription_Start_Site"] = pr_annotation["Start"]
pr_annotation = pr.PyRanges(pr_annotation)
gene_act, weigths = get_gene_activity(imputed_acc_obj, pr_annotation, chromsizes, use_gene_boundaries = True,
                                                              upstream = [1000, 100000], downstream = [1000,100000], distance_weight = True, 
                                                              decay_rate = 1, extend_gene_body_upstream = 10000, extend_gene_body_downstream = 500,
                                                              gene_size_weight = False, gene_size_scale_factor = "median", remove_promoters = False,
                                                              average_scores = True, scale_factor = 1, extend_tss = [10,10], gini_weight = True,
                                                              return_weights = True,  project = "Gene_activity")


RNA_seq = sc.read(os.path.join(workdir, "Gene_Cell.mtx"))
Cell_names = pd.read_csv(os.path.join(workdir, "Cell_names.tsv"), sep = "\t", header = None)
Gene_names = pd.read_csv(os.path.join(workdir, "Gene_names.tsv"), sep = "\t", header = None)
RNA_count = RNA_seq.X


adata =  ad.AnnData(RNA_count.transpose(), dtype = "int32")
adata.obs_names = Cell_names[0]
adata.var_names = Gene_names[0]
adata.var_names_make_unique()
adata.obs = cell_data.loc[adata.obs_names]
adata.obs_names = cistopic_obj.cell_names
adata.raw = adata
sc.pp.normalize_total(adata, target_sum = 1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, min_mean = 0.0125, max_mean = 3, min_disp = 0.5)
adata = adata[:, adata.var.highly_variable]
sc.pp.scale(adata, max_value = 10)
adata.write(os.path.join(out_dir, "adata.h5ad"))
pickle.dump(cistopic_obj, open(os.path.join(out_dir, "cistopic_obj.pkl"), "wb"))
pickle.dump(cistopic_obj, open(os.path.join(out_dir, "imputed_acc_obj.pkl"), "wb"))

In [5]:
import os
import mudata
from scenicplus.RSS import (regulon_specificity_scores, plot_rss)
from scenicplus.plotting.dotplot import heatmap_dotplot


out_dir = "./ComicGTN/data/ARPKD_kidney/SCENICplus_development_tutorial/outs"
mdata = mudata.read(os.path.join(out_dir, "ACC_GEX.h5mu"))
names = mdata["scATAC"].var_names.tolist()
new_names = [s.replace("-", ":", 1) for s in names]
mdata["scATAC"].var_names = new_names
mudata.write_h5mu(os.path.join(out_dir, "ACC_GEX.h5mu"), mdata)


scplus_mdata = mudata.read(os.path.join(out_dir, "scplusmdata.h5mu"))
scplus_mdata.uns["direct_e_regulon_metadata"].to_csv(os.path.join(out_dir, "eGRNs.csv"), index = False, header = True, sep = ",")
rss = regulon_specificity_scores(scplus_mudata = scplus_mdata, variable = "scRNA_counts:Cell_types",
                                                   modalities = ["direct_gene_based_AUC", "extended_gene_based_AUC"])
plot_rss(data_matrix = rss, top_n = 3, num_columns = 5)
sc.pl.umap(eRegulon_gene_AUC, color = list(set([x for xs in [rss.loc[ct].sort_values()[0:2].index for ct in rss.index] for x in xs ])))

heatmap_dotplot(scplus_mudata = scplus_mdata, color_modality = "direct_gene_based_AUC", size_modality = "direct_region_based_AUC",
                             group_variable = "scRNA_counts:Cell_types", eRegulon_metadata_key = "direct_e_regulon_metadata",
                             color_feature_key = "Gene_signature_name", size_feature_key = "Region_signature_name",
                             feature_name_key = "eRegulon_name", sort_data_by = "direct_gene_based_AUC", orientation = "vertical",
                             save = "./ComicGTN/data/ARPKD_kidney/heatmap.png", figsize = (30, 60))
scplus_mdata.uns["direct_e_regulon_metadata"].to_csv(os.path.join(out_dir, "eGRNs.csv"), index = False, header = True, sep = ",")